# Chroma 검색기 — 강사용 정답본

원본: `src/solution/Chapter01/03-rag/03_vector_store.py`  
대응 슬라이드: `docs/raw/Chapter01/day01/1-4-rag-theory-and-core-modules.pdf`

**학습 목표:** PDF 청킹 → Chroma → retriever → 상위 3개의 각 단계를 원본 코드에서 찾아 설명한다.

**실행 전:** 로컬 PDF를 읽어 청킹하고 임베딩 API를 호출해 Chroma를 메모리에 만든다. 질문을 계속 입력받으므로 중단할 때 커널을 멈춘다.

**특별 주의:** while True 입력 루프는 셀을 계속 붙잡는다. 중단할 때 커널을 멈춘다.



## 흐름을 먼저 읽기

PDF 청킹 → Chroma → retriever → 상위 3개

핵심은 문서 준비·검색·답변 단계의 경계이다. 입력이 어디서 들어오고 결과가 어디로 이동하는지 화살표를 따라가자. 원본 주석과 TODO를 그대로 둔다. TODO는 CodeMate에게 작업을 지시하고 결과를 검토할 지점으로 읽는다.


## CodeMate 실습 지시

> 다음 코드 흐름을 순서대로 설명해 주세요: PDF 청킹 → Chroma → retriever → 상위 3개. k=3이 답변 수와 검색 문서 수 중 무엇을 뜻하나? 원본 코드를 바꾸기 전에 근거가 되는 위치와 실행 위험을 알려 주세요.

CodeMate의 제안을 원본의 입력·출력·오류 흐름과 비교하고, 다른 결과가 나오면 다시 지시한다.


## 노트북 경로 설정

원본은 `__file__`로 PDF 위치를 찾는다. 노트북에서는 정의되지 않을 수 있으므로 원본 코드셀보다 먼저 이 설정 셀을 수동 실행한다. 프로젝트 루트를 찾지 못하면 명시적으로 오류가 난다. 원본 코드는 수정하지 않는다.


In [ ]:
from pathlib import Path
_cwd = Path.cwd().resolve()
_roots = [p for p in (_cwd, *_cwd.parents) if (p / 'pyproject.toml').is_file()]
if not _roots:
    raise FileNotFoundError('pyproject.toml이 있는 프로젝트 폴더에서 노트북을 열어 주세요.')
_project_root = _roots[0]
__file__ = str(_project_root / 'src/solution/Chapter01/03-rag/03_vector_store.py')
if not Path(__file__).is_file():
    raise FileNotFoundError(f'원본 Python 파일을 찾을 수 없습니다: {__file__}')


## 원본 코드 — 수동 실행

아래 셀은 `.py` 전체를 그대로 보존한다. 위에서 아래로 한 번에 실행되므로 환경과 부작용을 확인한다. **자동 실행/Run All을 사용하지 않는다.**

**이 코드셀의 실행 조건·부작용:** 로컬 PDF를 읽어 청킹하고 임베딩 API를 호출해 Chroma를 메모리에 만든다. 질문을 계속 입력받으므로 중단할 때 커널을 멈춘다.

**특별 주의:** while True 입력 루프는 셀을 계속 붙잡는다. 중단할 때 커널을 멈춘다.



In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import PDFMinerLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

from config.embeddings import get_embeddings

load_dotenv()


PDF_PATH = Path(__file__).parent / 'data' / 'meritz_pet_insurance.pdf'
loader = PDFMinerLoader(str(PDF_PATH.resolve()))
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
chunks = text_splitter.split_documents(documents)

# get_embeddings(): src/config/embeddings.py 에서 EMBED_PROVIDER 환경변수로 전환
# 로컬: OpenAIEmbeddings / 사내: SamsungEmbeddings (02_embeddings.py에서 직접 구현한 그것)
embeddings = get_embeddings()
vectorstore = Chroma.from_documents(
    chunks,
    embedding=embeddings,
    # ChromaDB를 저장하고 싶을 때
    # persist_directory=str(Path(__file__).parent),
)

retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
print('retriever ready')
while True:
    question = input('prompt: ')
    results = retriever.invoke(question)
    for index, doc in enumerate(results):
        text = ' '.join(doc.page_content.split())
        print(f'{index}. len={len(doc.page_content)} \n- {text[:180]}...')
        print()




## 결과를 검토하기

- 예상 동작: 질문을 입력하면 가까운 문서 청크를 최대 3개 출력한다.
- 확인 질문: k=3이 답변 수와 검색 문서 수 중 무엇을 뜻하나?
- 오류가 나면 먼저 원본 코드의 어느 단계인지 찾고, CodeMate가 제안한 수정의 이유와 부작용을 설명하게 한다.
